# OMNI-706 · Both encoders on Colab's GPU

Runs the repo's own encoders (`services/ai_workers/app/encoder.py` and `clip_encoder.py`), imported from a clone of the branch, never copied here.

1. **Setup:** clone the branch, install the pinned versions, mount Drive
2. **Load** SigLIP 2 (D-30) and InternVideo2-CLIP-S (D-33), both pinned, float32
3. **One clip and one text:** dimension, finite, unit length
4. **Does it see motion:** 10 VIRAT event clips of 5 types against 10 captions (a sanity check, not SM-1)
5. **GPU timings (D-31):** ms per frame / clip at batch 1, 8 and 32, ms per text, GPU memory

**Run on an A100:** Runtime → Change runtime type → A100 GPU. Then Runtime → Run all.

Everything it measures is written to `MyDrive/omnisight/results/omni706_encoders_gpu.json` at the end; send that file (or the last cell's output) back.

In [ ]:
# 1 · Setup
BRANCH = "slice-07-search"
REPO = "https://github.com/OmniSight-team/omni-sight.git"

import os, subprocess, sys
if not os.path.isdir("/content/omni-sight"):
    subprocess.run(["git", "clone", "-q", "-b", BRANCH, REPO, "/content/omni-sight"], check=True)
else:
    subprocess.run(["git", "-C", "/content/omni-sight", "pull", "-q"], check=True)
print(subprocess.run(["git", "-C", "/content/omni-sight", "log", "-1", "--format=%h %s"], capture_output=True, text=True).stdout)

# The versions in uv.lock; Colab keeps its own GPU build of torch and torchvision.
!pip install -q "transformers==5.20.1" "huggingface-hub==2.2.0" "safetensors==0.8.0" "open-clip-torch==3.3.0" -e /content/omni-sight/services/common

# The repo's code straight from the clone (an editable install is only seen after a
# runtime restart, so the paths are added here too).
sys.path.insert(0, "/content/omni-sight/services/common")
sys.path.insert(0, "/content/omni-sight/services/ai_workers")
from google.colab import drive
drive.mount("/content/drive", force_remount=True)

import torch
assert torch.cuda.is_available(), "no GPU: Runtime -> Change runtime type -> A100"
print(torch.__version__, torch.cuda.get_device_name(0))

In [ ]:
# 2 · Load both encoders, pinned (D-30, D-33), float32, on the GPU
import time
from pathlib import Path
from omnisight_common.settings import CommonSettings
from app.encoder import FrameTextEncoder, ensure_checkpoint
from app import clip_encoder
from app.clip_encoder import ClipTextEncoder

pins = {name: CommonSettings.model_fields[name].default for name in
        ("frame_model_id", "frame_model_revision", "clip_model_id", "clip_model_revision")}
print(pins)

DRIVE = Path("/content/drive/MyDrive/omnisight")
MODELS = DRIVE / "models" / "pretrained"   # cached on Drive, downloaded once at the pinned revision
VIRAT = DRIVE / "data" / "virat"
results = {"pins": pins, "gpu": torch.cuda.get_device_name(0), "torch": torch.__version__}

def gib(n): return round(n / 2**30, 2)

torch.cuda.reset_peak_memory_stats()
siglip_dir = ensure_checkpoint(pins["frame_model_id"], pins["frame_model_revision"], MODELS / "siglip2-so400m-patch14-384")
siglip = FrameTextEncoder(siglip_dir, device="cuda")
results["siglip2_weights_gib"] = gib(torch.cuda.memory_allocated())

before = torch.cuda.memory_allocated()
iv2_dir = ensure_checkpoint(pins["clip_model_id"], pins["clip_model_revision"], MODELS / "internvideo2-clip-s", clip_encoder.ALLOW_PATTERNS)
iv2 = ClipTextEncoder(iv2_dir, device="cuda")
results["internvideo2_weights_gib"] = gib(torch.cuda.memory_allocated() - before)
print(results)

In [ ]:
# 3 · One 8-frame clip and one text: the dimension, finite, unit length
# (the encoder itself refuses anything else; this shows the numbers)
import numpy as np
from omnisight_common.datasets import virat

name = "VIRAT_S_040103_00_000000_000120"
trunk = next(e for e in virat.read_events(VIRAT, name) if e.event_type == 3)
indices = clip_encoder.uniform_indices(trunk.start_frame, trunk.end_frame)
frames = clip_encoder.read_clip(VIRAT / "videos_original" / f"{name}.mp4", indices)
[c] = iv2.encode_clips([frames])
[t] = iv2.encode_texts(["a person opening the trunk of a car"])
results["one_clip"] = {
    "video": name, "event": trunk.event_id, "frames": indices,
    "clip_dim": int(c.shape[0]), "text_dim": int(t.shape[0]),
    "clip_norm": float(np.linalg.norm(c)), "text_norm": float(np.linalg.norm(t)),
    "finite": bool(np.isfinite(c).all() and np.isfinite(t).all()),
    "similarity": float(t @ c),
}
results["one_clip"]

## 4 · Does it see motion

10 clips, 2 of each of 5 event types, drawn once with seed 706 from the 275 VIRAT evaluation videos (`datasets/splits/virat.csv`), events at least 8 frames long. Each clip is the event's own span, 8 frames spread evenly over it (`uniform_indices`).

Two wordings per type, one per clip. Reported: how many clips rank **their own caption** first, and how many rank **a caption of their own type** first (the other wording of the same event is not a mistake).

In [ ]:
import csv, random
TYPES = [3, 4, 5, 6, 11]   # opening a trunk, closing a trunk, getting into / out of a vehicle, entering a facility
CAPTIONS = {
    3: ["a person opening the trunk of a car", "someone lifting open a car's trunk"],
    4: ["a person closing the trunk of a car", "someone shutting a car's trunk"],
    5: ["a person getting into a car", "someone climbing into a vehicle"],
    6: ["a person getting out of a car", "someone stepping out of a vehicle"],
    11: ["a person walking into a building", "someone entering a building through a door"],
}

with open("/content/omni-sight/datasets/splits/virat.csv") as f:
    videos = [row["clip_key"] for row in csv.DictReader(f) if row["split"] == "evaluation"]
candidates = {t: [] for t in TYPES}
for v in videos:
    if not (VIRAT / "annotations" / f"{v}.viratdata.events.txt").is_file():
        continue
    for e in virat.read_events(VIRAT, v):
        if e.event_type in candidates and e.end_frame - e.start_frame + 1 >= 8:
            candidates[e.event_type].append((v, e))
rng = random.Random(706)
chosen = [(t, *pair) for t in TYPES for pair in rng.sample(sorted(candidates[t], key=lambda p: (p[0], p[1].event_id)), 2)]

clips, captions, labels = [], [], []
for i, (t, v, e) in enumerate(chosen):
    clips.append(clip_encoder.read_clip(VIRAT / "videos_original" / f"{v}.mp4", clip_encoder.uniform_indices(e.start_frame, e.end_frame)))
    captions.append(CAPTIONS[t][i % 2])
    labels.append(t)

C = iv2.encode_clips(clips)
T = iv2.encode_texts(captions)
S = C @ T.T          # rows: clips, columns: captions
best = S.argmax(axis=1)
own = int(sum(best[i] == i for i in range(10)))
own_type = int(sum(labels[best[i]] == labels[i] for i in range(10)))
results["motion_check"] = {
    "clips": [{"video": v, "event": e.event_id, "type": t, "frames": [e.start_frame, e.end_frame]} for t, v, e in chosen],
    "captions": captions, "similarity": np.round(S, 4).tolist(),
    "own_caption_first": own, "own_type_first": own_type,
}
print("candidates per type:", {t: len(c) for t, c in candidates.items()})
print(f"own caption first: {own}/10 · a caption of its own type first: {own_type}/10")
import pandas as pd
pd.DataFrame(np.round(S, 3), index=[f"{virat.EVENT_TYPES[t][:24]} ({v[8:14]} e{e.event_id})" for t, v, e in chosen],
             columns=[f"c{j}" for j in range(10)])

## 5 · GPU timings (D-31)

Each figure: 5 warm-up calls, then 20 timed calls, `torch.cuda.synchronize()` around each. **Frames** are timed from the stored JPEG (what ai_workers does: read, resize, encode), **clips** from 8 decoded frames in memory (reading the video is not the encoder's cost). Peak GPU memory is per measurement, the weights included.

In [ ]:
import tempfile, statistics
from PIL import Image

def timed(fn, warm=5, runs=20):
    for _ in range(warm):
        fn()
    torch.cuda.synchronize()
    torch.cuda.reset_peak_memory_stats()
    times = []
    for _ in range(runs):
        torch.cuda.synchronize(); start = time.perf_counter()
        fn()
        torch.cuda.synchronize(); times.append((time.perf_counter() - start) * 1000)
    return statistics.median(times), gib(torch.cuda.max_memory_allocated())

# 32 real frames (the clips above), as the full-frame JPEGs ai_workers reads
scratch = Path(tempfile.mkdtemp())
paths = []
for i, frame in enumerate(f for clip in clips for f in clip):
    if i == 32: break
    p = scratch / f"{i}.jpg"; Image.fromarray(frame).save(p, quality=90); paths.append(p)
clip32 = (clips * 4)[:32]
query = ["a person opening the trunk of a car"]

timings = {}
for batch in (1, 8, 32):
    ms, mem = timed(lambda: siglip.encode_images(paths[:batch]))
    timings[f"siglip2_image_b{batch}"] = {"ms_per_call": round(ms, 1), "ms_per_item": round(ms / batch, 2), "peak_gib": mem}
    ms, mem = timed(lambda: iv2.encode_clips(clip32[:batch]))
    timings[f"internvideo2_clip_b{batch}"] = {"ms_per_call": round(ms, 1), "ms_per_item": round(ms / batch, 2), "peak_gib": mem}
ms, mem = timed(lambda: siglip.encode_texts(query))
timings["siglip2_text_b1"] = {"ms_per_call": round(ms, 2), "peak_gib": mem}
ms, mem = timed(lambda: iv2.encode_texts(query))
timings["internvideo2_text_b1"] = {"ms_per_call": round(ms, 2), "peak_gib": mem}
results["timings"] = timings
pd.DataFrame(timings).T

In [ ]:
# Save everything to Drive
import json
out = DRIVE / "results" / "omni706_encoders_gpu.json"
out.parent.mkdir(parents=True, exist_ok=True)
out.write_text(json.dumps(results, indent=2))
print(out)
print(json.dumps(results, indent=2))